In [1]:
import torch
import torch.nn as nn 
import os
import pandas as pd
from tqdm import tqdm
import csv
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

from transformers import CLIPModel, CLIPProcessor
from peft import LoraConfig, get_peft_model, TaskType

/home/arnavw/Documents/amazon-ml-2025/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class PriceDataset(Dataset):
    def __init__(self, img_path, image_paths, item_names, bullet_points, 
                 product_descriptions, values, units, prices, processor, max_length=77):
        self.img_dir = img_path
        self.image_paths = image_paths
        self.item_names = item_names
        self.bullet_points = bullet_points
        self.product_descriptions = product_descriptions
        self.values = values
        self.units = units
        self.prices = torch.tensor(prices, dtype=torch.float32)
        self.processor = processor
        self.max_length = max_length
    
    def __len__(self):
        return len(self.image_paths)
    
    def _concatenate_text(self, idx):
        """
        Concatenate all text fields into one string.
        Prioritizes most important information first (for token limit).
        """
        text_parts = []
        
        # Item Name (most important)
        if pd.notna(self.item_names[idx]) and str(self.item_names[idx]).strip():
            text_parts.append(str(self.item_names[idx]).strip())
        
        # Quantity (concise and important for pricing)
        if pd.notna(self.values[idx]) and pd.notna(self.units[idx]):
            text_parts.append(f"{self.values[idx]} {self.units[idx]}")
        
        # Bullet Points (key features)
        if pd.notna(self.bullet_points[idx]) and str(self.bullet_points[idx]).strip():
            text_parts.append(str(self.bullet_points[idx]).strip())
        
        # Product Description (detailed, might be truncated)
        if pd.notna(self.product_descriptions[idx]) and str(self.product_descriptions[idx]).strip():
            text_parts.append(str(self.product_descriptions[idx]).strip())
        
        # Join with separator that CLIP handles well
        full_text = ". ".join(text_parts)
        
        # Handle empty text case
        if not full_text.strip():
            full_text = "product"  # Fallback text
        
        return full_text
    
    def __getitem__(self, idx):
        """
        Returns a single sample with image and text features.
        """
        # Load image
        img_filename = f"{self.image_paths[idx]}.jpg"
        img_path = os.path.join(self.img_dir, img_filename)
        
        try:
            img = Image.open(img_path).convert('RGB')
        except Exception as e:
            # Return a blank image as fallback
            img = Image.new('RGB', (224, 224), color='white')
        
        # Concatenate text fields
        full_text = self._concatenate_text(idx)
        
        # Process with CLIP processor
        inputs = self.processor(
            text=full_text,
            images=img,
            return_tensors="pt",
            padding="max_length",  # Consistent padding
            truncation=True,
            max_length=self.max_length
        )
        
        return {
            'pixel_values': inputs['pixel_values'].squeeze(0),
            'input_ids': inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0),
            'price': self.prices[idx]
        }

In [3]:
class SMAPELoss(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps
    
    def forward(self, pred, target):
        # Ensure inputs are valid
        pred = torch.clamp(pred, min=-1e6, max=1e6)  # Prevent extreme values
        target = torch.clamp(target, min=-1e6, max=1e6)
        
        numerator = torch.abs(pred - target)
        denominator = (torch.abs(pred) + torch.abs(target) + self.eps)
        
        # Prevent division by zero
        denominator = torch.clamp(denominator, min=self.eps)
        
        smape = 2.0 * numerator / denominator
        
        # Check for NaN/inf before returning
        smape = torch.where(torch.isnan(smape) | torch.isinf(smape), 
                           torch.zeros_like(smape), smape)
        
        return torch.mean(smape) * 100

In [11]:
class MultimodalSelfAttention(nn.Module):
    def __init__(self, feature_dim, num_heads=8, dropout=0.1):
        super().__init__()
        self.feature_dim = feature_dim
        self.num_heads = num_heads
        self.head_dim = feature_dim // num_heads
        
        assert feature_dim % num_heads == 0, "feature_dim must be divisible by num_heads"
        
        # Linear projections for Q, K, V
        self.query = nn.Linear(feature_dim, feature_dim)
        self.key = nn.Linear(feature_dim, feature_dim)
        self.value = nn.Linear(feature_dim, feature_dim)
        
        # Output projection
        self.out_proj = nn.Linear(feature_dim, feature_dim)
        
        # Dropout and normalization
        self.dropout = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(feature_dim)
        
        # Initialize weights
        self._init_weights()
    
    def _init_weights(self):
        for module in [self.query, self.key, self.value, self.out_proj]:
            nn.init.xavier_uniform_(module.weight)
            nn.init.zeros_(module.bias)
    
    def forward(self, image_features, text_features):
        """
        Apply self-attention across image and text features.
        
        Args:
            image_features: [batch_size, feature_dim]
            text_features: [batch_size, feature_dim]
        
        Returns:
            fused_features: [batch_size, feature_dim * 2]
        """
        batch_size = image_features.size(0)
        
        # Stack features for multimodal attention [batch_size, 2, feature_dim]
        multimodal_features = torch.stack([image_features, text_features], dim=1)
        
        # Compute Q, K, V
        Q = self.query(multimodal_features)  # [batch_size, 2, feature_dim]
        K = self.key(multimodal_features)
        V = self.value(multimodal_features)
        
        # Reshape for multi-head attention
        Q = Q.view(batch_size, 2, self.num_heads, self.head_dim).transpose(1, 2)  # [batch_size, num_heads, 2, head_dim]
        K = K.view(batch_size, 2, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, 2, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Compute attention scores
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)  # [batch_size, num_heads, 2, 2]
        attention_weights = torch.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        
        # Apply attention to values
        attended_features = torch.matmul(attention_weights, V)  # [batch_size, num_heads, 2, head_dim]
        
        # Concatenate heads and reshape
        attended_features = attended_features.transpose(1, 2).contiguous().view(
            batch_size, 2, self.feature_dim
        )  # [batch_size, 2, feature_dim]
        
        # Apply output projection
        attended_features = self.out_proj(attended_features)
        
        # Residual connection and layer norm
        attended_features = self.layer_norm(attended_features + multimodal_features)
        
        # Flatten to get final fused representation
        fused_features = attended_features.view(batch_size, -1)  # [batch_size, feature_dim * 2]
        
        return fused_features


class CLIPPricePredictorAttention(nn.Module):
    def __init__(self, model_name="openai/clip-vit-base-patch32", num_attention_heads=8):
        super().__init__()
        # Load CLIP model
        self.clip = CLIPModel.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
            device_map="cuda"
        )
        
        # Freeze CLIP parameters
        for param in self.clip.parameters():
            param.requires_grad = False
        
        # Get CLIP projection dimension (typically 512 for base models)
        clip_dim = self.clip.config.projection_dim
        model_device = next(self.clip.parameters()).device
        
        # Self-attention based multimodal fusion
        self.multimodal_attention = MultimodalSelfAttention(
            feature_dim=clip_dim,
            num_heads=num_attention_heads,
            dropout=0.1
        )
        
        # Regression head for fused multimodal features
        self.regressor = nn.Sequential(
            nn.LayerNorm(clip_dim * 2),
            nn.Linear(clip_dim * 2, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            
            nn.Linear(128, 1),
            nn.ReLU()
        )
        
        self._init_weights()
        
        # Move components to device
        self.multimodal_attention.to(model_device, dtype=torch.float32)
        self.regressor.to(model_device, dtype=torch.float32)
    
    def _init_weights(self):
        for layer in self.regressor:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_normal_(layer.weight, gain=0.01)
                nn.init.zeros_(layer.bias)
    
    @torch.no_grad()
    def extract_features(self, pixel_values, input_ids, attention_mask):
        """
        Extract frozen multimodal embeddings from CLIP.
        """
        self.clip.eval()
        
        # Get CLIP outputs
        outputs = self.clip(
            pixel_values=pixel_values,
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )
        
        # Extract normalized embeddings from CLIP's projection space
        image_embeds = outputs.image_embeds  # [batch_size, projection_dim]
        text_embeds = outputs.text_embeds    # [batch_size, projection_dim]
        
        # Clamp for stability
        image_embeds = torch.clamp(image_embeds.float(), min=-5, max=5)
        text_embeds = torch.clamp(text_embeds.float(), min=-5, max=5)
        
        return image_embeds, text_embeds
    
    def forward(self, pixel_values, input_ids, attention_mask=None):
        """
        Forward pass for price prediction with self-attention fusion.
        
        Args:
            pixel_values: Preprocessed images [batch_size, 3, H, W]
            input_ids: Tokenized text [batch_size, seq_len]
            attention_mask: Attention mask for text [batch_size, seq_len]
        
        Returns:
            price_pred: Predicted prices [batch_size]
        """
        # Extract features from CLIP
        with torch.no_grad():
            image_features, text_features = self.extract_features(pixel_values, input_ids, attention_mask)
        
        # Apply self-attention based multimodal fusion
        fused_features = self.multimodal_attention(image_features, text_features)
        
        # Predict price using regression head
        price_pred = self.regressor(fused_features)
        return price_pred.squeeze(-1)

In [5]:
class CLIPPricePredictor(nn.Module):
    def __init__(self, model_name="openai/clip-vit-base-patch32"):
        super().__init__()
        # Load CLIP model
        self.clip = CLIPModel.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
            device_map="cuda"
        )
        
        # Freeze CLIP parameters
        for param in self.clip.parameters():
            param.requires_grad = False
        
        # Get CLIP projection dimension (typically 512 for base models)
        clip_dim = self.clip.config.projection_dim
        model_device = next(self.clip.parameters()).device
        
        # Regression head for multimodal features (concatenation fusion)
        self.regressor = nn.Sequential(
            nn.LayerNorm(clip_dim * 2),
            nn.Linear(clip_dim * 2, 256),
            nn.ReLU(),
            
            nn.Dropout(0.2),
            nn.Linear(256, 1),
            nn.ReLU()
        )
        
        self._init_weights()
        self.regressor.to(model_device, dtype=torch.float32)
    
    def _init_weights(self):
        for layer in self.regressor:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_normal_(layer.weight, gain=0.01)
                nn.init.zeros_(layer.bias)
    
    @torch.no_grad()
    def extract_features(self, pixel_values, input_ids, attention_mask):
        """
        Extract frozen multimodal embeddings from CLIP.
        Combines image and text features in the shared embedding space.
        """
        self.clip.eval()
        
        # Get CLIP outputs
        outputs = self.clip(
            pixel_values=pixel_values,
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )
        
        # Extract normalized embeddings from CLIP's projection space
        image_embeds = outputs.image_embeds  # [batch_size, projection_dim]
        text_embeds = outputs.text_embeds    # [batch_size, projection_dim]
        
        # Concatenate image and text features (concatenation fusion)
        multimodal_features = torch.cat([image_embeds, text_embeds], dim=-1)
        
        # Clamp for stability
        return torch.clamp(multimodal_features.float(), min=-5, max=5)
    
    def forward(self, pixel_values, input_ids, attention_mask=None):
        """
        Forward pass for price prediction.
        
        Args:
            pixel_values: Preprocessed images [batch_size, 3, H, W]
            input_ids: Tokenized text [batch_size, seq_len]
            attention_mask: Attention mask for text [batch_size, seq_len]
        
        Returns:
            price_pred: Predicted prices [batch_size]
        """
        with torch.no_grad():
            features = self.extract_features(pixel_values, input_ids, attention_mask)
        
        price_pred = self.regressor(features)
        return price_pred.squeeze(-1)

In [6]:
def apply_qlora(model, r=16, alpha=32):
    lora_config = LoraConfig(
        r=r,
        lora_alpha=alpha,
        target_modules=[
            "query", "key", "value",
            "attention.self.query",
            "attention.self.key", 
            "attention.self.value",
            "crossattention.self.query",
            "crossattention.self.key",
            "crossattention.self.value"
        ],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.FEATURE_EXTRACTION,
    )
    
    model.blip2.qformer = get_peft_model(model.blip2.qformer, lora_config)
    
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
    
    return model

In [12]:
def create_enhanced_model(model_name="openai/clip-vit-base-patch32", num_attention_heads=8):
    model = CLIPPricePredictorAttention(
        model_name=model_name,
        num_attention_heads=num_attention_heads
    )
    
    # Print model architecture
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print(f"Model Architecture Summary:")
    print(f"├── CLIP Model: {model_name}")
    print(f"├── Attention Heads: {num_attention_heads}")
    print(f"├── Total Parameters: {total_params:,}")
    print(f"├── Trainable Parameters: {trainable_params:,}")
    print(f"└── Trainable Ratio: {100*trainable_params/total_params:.2f}%")
    
    return model

In [8]:
def train(model, train_loader, val_loader, epochs=10, lr=1e-4, device='cuda', log_file='training_log.csv', early_stopping_patience=3, scheduler_step_interval=500):
    model.to(device)
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr,
        weight_decay=1e-2
    )
    
    # Use MSE or Huber loss for regression
    criterion = nn.MSELoss()
    
    # Simple cosine annealing scheduler
    scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2)
    
    # Setup logging
    if not os.path.exists(log_file):
        with open(log_file, 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(['Epoch', 'Train_Loss', 'Val_Loss', 'LR'])
    
    best_val_loss = float('inf')
    best_epoch = 0
    early_stopping_counter = 0
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        prev_lr = optimizer.param_groups[0]['lr']
        
        train_pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs} [Train]')
        for batch_idx, batch in enumerate(train_pbar):
            # Move data to device with correct dtypes
            pixel_values = batch['pixel_values'].to(device, dtype=torch.float16, non_blocking=True)
            input_ids = batch['input_ids'].to(device, non_blocking=True)
            attention_mask = batch['attention_mask'].to(device, non_blocking=True)
            prices = batch['price'].to(device, dtype=torch.float32, non_blocking=True)
            
            # Forward pass
            optimizer.zero_grad()
            predictions = model(pixel_values, input_ids, attention_mask)
            
            # Compute loss
            loss = criterion(predictions, prices)
            
            # Backward pass
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            # Step the scheduler every scheduler_step_interval batches
            if (batch_idx + 1) % scheduler_step_interval == 0:
                scheduler.step(epoch + (batch_idx + 1) / len(train_loader))
                new_lr = optimizer.param_groups[0]['lr']
                print(f"Scheduler stepped at batch {batch_idx+1}: LR changed from {prev_lr:.6e} to {new_lr:.6e} (Δ={(new_lr-prev_lr):.2e})")
                prev_lr = new_lr
            
            train_loss += loss.item()
            train_pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'lr': f'{optimizer.param_groups[0]["lr"]:.2e}'
            })
        
        # Final scheduler step at end of epoch if not already stepped
        if (batch_idx + 1) % scheduler_step_interval != 0:
            scheduler.step(epoch + 1)
            new_lr = optimizer.param_groups[0]['lr']
            print(f"Scheduler stepped at end of epoch: LR changed from {prev_lr:.6e} to {new_lr:.6e} (Δ={(new_lr-prev_lr):.2e})")
        
        avg_train_loss = train_loss / len(train_loader)
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        
        val_pbar = tqdm(val_loader, desc=f'Epoch {epoch+1}/{epochs} [Val]', leave=False)
        with torch.no_grad():
            for batch in val_pbar:
                pixel_values = batch['pixel_values'].to(device, dtype=torch.float16, non_blocking=True)
                input_ids = batch['input_ids'].to(device, non_blocking=True)
                attention_mask = batch['attention_mask'].to(device, non_blocking=True)
                prices = batch['price'].to(device, dtype=torch.float32, non_blocking=True)
                
                predictions = model(pixel_values, input_ids, attention_mask)
                loss = criterion(predictions, prices)
                
                val_loss += loss.item()
                val_pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        avg_val_loss = val_loss / len(val_loader)
        
        # Log metrics
        current_lr = optimizer.param_groups[0]['lr']
        with open(log_file, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([epoch + 1, avg_train_loss, avg_val_loss, current_lr])
        
        print(f'Epoch {epoch+1}/{epochs} - Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, LR: {current_lr:.2e}')
        
        # Early Stopping logic
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_epoch = epoch + 1
            early_stopping_counter = 0
            torch.save(model.state_dict(), 'best_model.pth')
            print(f'✓ Saved best model with val_loss: {best_val_loss:.4f}')
        else:
            early_stopping_counter += 1
            print(f'Early stopping patience: {early_stopping_counter}/{early_stopping_patience}')
            if early_stopping_counter >= early_stopping_patience:
                print(f'\nEarly stopping triggered at epoch {epoch+1}. Best validation loss: {best_val_loss:.4f} (epoch {best_epoch})')
                break
        
        # Clear cache periodically
        if (epoch + 1) % 5 == 0:
            torch.cuda.empty_cache()
    
    print(f'\nTraining complete! Best validation loss: {best_val_loss:.4f} (epoch {best_epoch})')
    return model

In [9]:
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


## Self-Attention Based Multimodal Fusion

### Key Improvements:

1. **MultimodalSelfAttention Module**: 
   - Replaces simple concatenation with learnable attention mechanism
   - Allows dynamic weighting between image and text features
   - Multi-head attention for capturing different aspects of cross-modal relationships

2. **Enhanced Architecture**:
   - Deeper regression head (4 layers vs 2) for better feature transformation
   - Residual connections and layer normalization in attention module
   - Improved dropout scheduling for better regularization

3. **Benefits**:
   - **Adaptive Fusion**: Model learns how much to rely on each modality per sample
   - **Cross-Modal Interactions**: Attention mechanism captures dependencies between image and text
   - **Representation Learning**: Self-attention creates richer joint representations
   - **Scalability**: Can easily extend to more modalities if needed

4. **Technical Details**:
   - 8-head attention by default (configurable)
   - Xavier initialization for stable training
   - Gradient clipping compatible
   - Memory efficient implementation

In [13]:
TRAIN_IMG_DIR = '../images/train_part1'
VAL_IMG_DIR = '../images/val_part1'
train_df = pd.read_csv('../dataset/train_split/train_part1.csv')
val_df = pd.read_csv('../dataset/val_split/val_part1.csv') 

train_dl = DataLoader(
    PriceDataset(
        img_path = TRAIN_IMG_DIR,
        image_paths=train_df['sample_id'].values,
        item_names=train_df['Item Name'].values,
        bullet_points=train_df['Bullet Points'].values,
        product_descriptions=train_df['Product Description'].values,
        values=train_df['Value'].values,
        units=train_df['Unit'].values,
        prices=train_df['log_price'].values,  
        processor=processor
    ),
    batch_size=4,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)
val_dl = DataLoader(
    PriceDataset(
        img_path = VAL_IMG_DIR,
        image_paths=val_df['sample_id'].values,
        item_names=val_df['Item Name'].values,
        bullet_points=val_df['Bullet Points'].values,
        product_descriptions=val_df['Product Description'].values,
        values=val_df['Value'].values,
        units=val_df['Unit'].values,
        prices=val_df['log_price'].values,  
        processor=processor
    ),
    batch_size=4,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)    

# Create enhanced model with self-attention fusion
model = create_enhanced_model(
    model_name="openai/clip-vit-base-patch32",
    num_attention_heads=8
)
    
model = train(model, train_dl, val_dl, epochs=3, lr=1.29e-4)
print("Done!")

`torch_dtype` is deprecated! Use `dtype` instead!


Model Architecture Summary:
├── CLIP Model: openai/clip-vit-base-patch32
├── Attention Heads: 8
├── Total Parameters: 153,020,162
├── Trainable Parameters: 1,742,849
└── Trainable Ratio: 1.14%


Epoch 1/3 [Train]:  10%|█         | 501/4898 [01:01<09:15,  7.92it/s, loss=0.8448, lr=1.29e-04]

Scheduler stepped at batch 500: LR changed from 1.290000e-04 to 1.288674e-04 (Δ=-1.33e-07)


Epoch 1/3 [Train]:  20%|██        | 1001/4898 [01:40<05:58, 10.86it/s, loss=0.6300, lr=1.28e-04]

...

Epoch 3/3 [Train]:  92%|█████████▏| 4501/4898 [06:23<00:26, 15.18it/s, loss=0.8561, lr=4.77e-05]

Scheduler stepped at batch 4500: LR changed from 5.175137e-05 to 4.772493e-05 (Δ=-4.03e-06)


Epoch 3/3 [Train]: 100%|██████████| 4898/4898 [07:03<00:00, 11.56it/s, loss=0.0119, lr=4.77e-05]


Scheduler stepped at end of epoch: LR changed from 4.772493e-05 to 4.456840e-05 (Δ=-3.16e-06)


Epoch 3/3 - Train Loss: 0.6314, Val Loss: 0.6076, LR: 4.46e-05
✓ Saved best model with val_loss: 0.6076

Training complete! Best validation loss: 0.6076 (epoch 3)
Done!


### Training over Parts 2 and 3

In [14]:
del train_dl, val_dl, train_df, val_df  # Free up memory
import gc
gc.collect()
torch.cuda.empty_cache()

In [15]:
train2_df = pd.read_csv('../dataset/train_split/part2.csv')
val2_df = pd.read_csv('../dataset/val_split/part2.csv')

In [ ]:
train2_dl = DataLoader(
    PriceDataset(
        img_path='../images/train_part2',
        image_paths=train2_df['sample_id'].values,
        item_names=train2_df['Item Name'].values,
        bullet_points=train2_df['Bullet Points'].values,
        product_descriptions=train2_df['Product Description'].values,
        values=train2_df['Value'].values,
        units=train2_df['Unit'].values,
        prices=train2_df['log_price'].values,
        processor=processor
    ),
    batch_size=4,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)
val2_dl = DataLoader(
    PriceDataset(
        img_path='../images/val_part2',
        image_paths=val2_df['sample_id'].values,
        item_names=val2_df['Item Name'].values,
        bullet_points=val2_df['Bullet Points'].values,
        product_descriptions=val2_df['Product Description'].values,
        values=val2_df['Value'].values,
        units=val2_df['Unit'].values,
        prices=val2_df['log_price'].values,
        processor=processor
    ),
    batch_size=4,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

model = train(model, train2_dl, val2_dl, epochs=3, lr=5e-5, log_file='training_log_part2.csv')


Epoch 1/3 [Train]:  10%|█         | 499/4939 [00:41<06:21, 11.63it/s, loss=1.0602, lr=4.99e-05]

Scheduler stepped at batch 500: LR changed from 5.000000e-05 to 4.994944e-05 (Δ=-5.06e-08)


Epoch 1/3 [Train]:  20%|██        | 1000/4939 [01:19<04:29, 14.59it/s, loss=0.1938, lr=4.98e-05]

Scheduler stepped at batch 1000: LR changed from 4.994944e-05 to 4.979797e-05 (Δ=-1.51e-07)


Epoch 1/3 [Train]:  30%|███       | 1501/4939 [01:56<04:07, 13.88it/s, loss=0.5984, lr=4.95e-05]

...

Epoch 3/3 [Train]:  91%|█████████ | 4500/4939 [06:24<00:33, 13.25it/s, loss=0.1580, lr=1.86e-05]

Scheduler stepped at batch 4500: LR changed from 2.016310e-05 to 1.861379e-05 (Δ=-1.55e-06)


Epoch 3/3 [Train]: 100%|██████████| 4939/4939 [07:01<00:00, 11.73it/s, loss=0.2773, lr=1.86e-05]


Scheduler stepped at end of epoch: LR changed from 1.861379e-05 to 1.727458e-05 (Δ=-1.34e-06)


Epoch 3/3 - Train Loss: 0.5432, Val Loss: 0.5632, LR: 1.73e-05
✓ Saved best model with val_loss: 0.5632

Training complete! Best validation loss: 0.5632 (epoch 3)


In [16]:
del train2_dl, val2_dl, train2_df, val2_df  # Free up memory
import gc
gc.collect()
torch.cuda.empty_cache()

NameError: name 'train2_dl' is not defined

In [17]:
train3_df = pd.read_csv('../dataset/train_split/part3.csv')
val3_df = pd.read_csv('../dataset/val_split/part3.csv')


train3_dl = DataLoader(
    PriceDataset(
        img_path='../images/train_part3',
        image_paths=train3_df['sample_id'].values,
        item_names=train3_df['Item Name'].values,
        bullet_points=train3_df['Bullet Points'].values,
        product_descriptions=train3_df['Product Description'].values,
        values=train3_df['Value'].values,
        units=train3_df['Unit'].values,
        prices=train3_df['log_price'].values,
        processor=processor
    ),
    batch_size=4,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)
val3_dl = DataLoader(
    PriceDataset(
        img_path='../images/val_part3',
        image_paths=val3_df['sample_id'].values,
        item_names=val3_df['Item Name'].values,
        bullet_points=val3_df['Bullet Points'].values,
        product_descriptions=val3_df['Product Description'].values,
        values=val3_df['Value'].values,
        units=val3_df['Unit'].values,
        prices=val3_df['log_price'].values,
        processor=processor
    ),
    batch_size=4,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

model = train(model, train3_dl, val3_dl, epochs=3, lr=1.18e-5, log_file='training_log_part3.csv')

torch.save({
    'regressor': model.regressor.state_dict(),
    'model_name': model.clip.config._name_or_path,
    'clip_dim': model.clip.config.projection_dim,
}, 'model_final_3.pth')

print("Training on part 3 complete!")

Epoch 1/3 [Train]:  10%|█         | 501/4996 [00:56<06:11, 12.09it/s, loss=2.7605, lr=1.18e-05]

Scheduler stepped at batch 500: LR changed from 1.180000e-05 to 1.178834e-05 (Δ=-1.17e-08)


Epoch 1/3 [Train]:  20%|█▉        | 999/4996 [01:43<08:41,  7.67it/s, loss=0.5887, lr=1.18e-05]

Scheduler stepped at batch 1000: LR changed from 1.178834e-05 to 1.175340e-05 (Δ=-3.49e-08)


Epoch 1/3 [Train]:  30%|███       | 1500/4996 [02:24<05:47, 10.07it/s, loss=1.4811, lr=1.17e-05]

...

Epoch 3/3 [Train]:  90%|█████████ | 4500/4996 [05:31<00:39, 12.59it/s, loss=0.5487, lr=4.43e-06]

Scheduler stepped at batch 4500: LR changed from 4.792118e-06 to 4.430143e-06 (Δ=-3.62e-07)


Epoch 3/3 [Train]: 100%|██████████| 4996/4996 [06:10<00:00, 13.48it/s, loss=0.5322, lr=4.43e-06]


Scheduler stepped at end of epoch: LR changed from 4.430143e-06 to 4.076800e-06 (Δ=-3.53e-07)


Epoch 3/3 - Train Loss: 0.5980, Val Loss: 0.5633, LR: 4.08e-06
✓ Saved best model with val_loss: 0.5633

Training complete! Best validation loss: 0.5633 (epoch 3)
Training on part 3 complete!
